In [1]:
# Code describing the various ways that data can be loaded into the quantum environment
# Then, a quantum random walk algorithm is used to project values forward to illustrate uses
# 1) Binary Encoding
# Code and process developed by: Dr. Michael P. Haydock - IBM Fellow Emeritus, Visiting Professor at St. Olaf College
# Initial Coding: 3/4/2025

# Load the libraries
import pandas as pd  # Import pandas for data manipulation
import numpy as np  # Import numpy for numerical operations
import pennylane as qml  # Import PennyLane for quantum computing

In [2]:

# Load data
file_path = "economic_data.csv"  # Replace with your actual file path
data = pd.read_csv(file_path)  # Read the CSV file into a pandas DataFrame

# Drop the 'Date' column and use only numerical columns
numeric_data = data.iloc[:, 1:]  # Select all columns to the right of 'Date'

# Normalize the data to range [0, 1]
normalized_data = (numeric_data - numeric_data.min()) / (numeric_data.max() - numeric_data.min())

# Validate the normalized data
if normalized_data.isnull().values.any():  # Check for missing (NaN) values after normalization
    raise ValueError("Unexpected NaN values after normalization. Check numeric ranges.")

# Dynamically set the number of qubits
num_qubits = len(numeric_data.columns)  # Match the number of qubits to the number of columns
dev = qml.device("default.qubit", wires=num_qubits)  # Define the quantum device with required qubits


FileNotFoundError: [Errno 2] No such file or directory: 'economic_data.csv'

In [ ]:


# Quantum binary encoding function
def binary_encoding(data):
    """
    Encode data into binary using PauliX gates:
    - If a value is greater than 0.5, it is set as 1 and encoded with a PauliX gate.
    - Otherwise, it remains as 0.
    """
    binary_data = (data > 0.5).astype(int)  # Convert normalized values to binary
    for idx, value in enumerate(binary_data):
        if value == 1:  # Apply a PauliX gate for binary '1'
            qml.PauliX(wires=idx)

# Define a single quantum random walk step
def random_walk_step():
    """
    Add variability and dynamics to the quantum walk:
    - Hadamard gates create superposition.
    - Randomized RY, RZ, and CRX gates introduce variability and entanglement.
    """
    for wire in range(num_qubits):
        qml.Hadamard(wires=wire)  # Create superposition on each qubit
        qml.RY(np.random.uniform(0, np.pi), wires=wire)  # Randomize the angle of the RY gate
    for i in range(num_qubits - 1):
        qml.CRX(np.random.uniform(0, np.pi / 2), wires=[i, i + 1])  # Add random entanglement between qubits
    qml.RZ(np.random.uniform(0, np.pi), wires=num_qubits - 1)  # Final random rotation on the last qubit

# Define the quantum node for performing the random walk
@qml.qnode(dev)
def random_walk(data):
    """
    Perform a single step of the quantum random walk:
    - Binary encoding initializes the state.
    - Random walk dynamics are applied multiple times.
    - Returns the probabilities of each computational basis state.
    """
    binary_encoding(data)  # Encode the input data as binary
    for _ in range(3):  # Apply the random walk step three times for greater dynamics
        random_walk_step()
    return qml.probs(wires=range(num_qubits))  # Return the probabilities for each state

# Forecast function for future time periods
def forecast(data, steps=12):
    """
    Generate predictions for future time periods:
    - Start with the last row of the input data as the initial state.
    - Apply quantum random walk repeatedly to predict values.
    - Rescale predictions back to their original range.
    """
    input_vector = data.iloc[-1].values  # Take the last row of normalized data as the starting point
    predictions = []  # List to store forecasted values

    for _ in range(steps):  # Perform the forecast for the specified number of time steps
        probabilities = random_walk(input_vector)  # Perform the quantum walk
        next_values = probabilities[:len(data.columns)]  # Use only the relevant probabilities
        # Rescale probabilities to match the original data range
        next_values = next_values * (numeric_data.max() - numeric_data.min()) + numeric_data.min()
        predictions.append(next_values)  # Append forecasted values to the list

        # Re-normalize with slight noise to add variability in subsequent predictions
        noise = np.random.uniform(-0.01, 0.01, size=next_values.shape)
        input_vector = ((next_values + noise) - numeric_data.min().values) / (numeric_data.max().values - numeric_data.min().values)

    # Convert the predictions into a pandas DataFrame with appropriate column names
    return pd.DataFrame(predictions, columns=numeric_data.columns)

# Generate the 12-step forecast
forecasted_data = forecast(normalized_data)  # Call the forecast function on the normalized data
forecasted_data.index = [f"Forecast {i+1}" for i in range(12)]  # Label the forecasted rows

# Visualize the quantum circuit for the first input
input_vector = normalized_data.iloc[-1].values  # Use the last row of normalized data for visualization
drawer = qml.draw(random_walk)  # Prepare to draw the quantum circuit
circuit_diagram = drawer(input_vector)  # Generate the circuit diagram
print("Quantum Circuit for the Enhanced Random Walk:")
print(circuit_diagram)  # Print the quantum circuit diagram

# Display the forecasted data
print("\n12-Time-Period Forecast:")
print(forecasted_data)  # Print the forecasted data as a DataFrame